# ⚡ Deepwoken AI Build Analyzer (Google Colab 24시간 클라우드)
> **구글 클라우드 무료 서버에서 여러 개의 유튜브 영상, 구글 닥스, 웹 가이드를 한 번에 자동 분석합니다.**
- 📱 PC/스마트폰/태블릿 어디서든 웹 브라우저로 실행 가능
- 💤 실행 버튼 누르고 브라우저나 컴퓨터를 꺼도 클라우드에서 끝까지 안전하게 완료됩니다.

In [ ]:
# 1. ⚙️ 필수 라이브러리 및 AI 엔진 설치 (▶ 클릭하여 실행)
!pip install -q google-generativeai yt-dlp pydantic python-dotenv rich chromadb beautifulsoup4 pyyaml imageio-ffmpeg
!apt-get -qq install -y ffmpeg > /dev/null 2>&1

import os, sys, re, json, time, urllib.request
from bs4 import BeautifulSoup
import google.generativeai as genai
import yt_dlp

os.makedirs("data/analysis", exist_ok=True)
os.makedirs("data/knowledge_base", exist_ok=True)
os.makedirs("data/videos", exist_ok=True)

print("✅ 구글 클라우드 AI 분석 환경 준비 완료!")

In [ ]:
# 2. 🚀 분석할 링크 목록 및 실행 (▶ 클릭하여 실행)

# 🔑 [설정 1] 발급받으신 Gemini API 키를 따옴표 안에 넣으세요 (여러 개일 경우 쉼표로 연결)
API_KEYS = "" 

# 📋 [설정 2] 분석할 링크 목록 (자유롭게 추가/수정 가능)
URL_LIST = [
    "https://www.youtube.com/watch?v=sJm04d99BK8",
    "https://www.youtube.com/watch?v=hOPe91tv5BA",
    "https://www.youtube.com/watch?v=LldsffJqtCY",
    "https://www.youtube.com/watch?v=_ihaDl5m6Rs",
    "https://docs.google.com/document/d/1Vc--1fU8IXyOn9zesPSijBK7W0J7Z-7FZbTx_-GFBFc/edit?tab=t.0",
    "https://docs.google.com/document/d/1Up1ZpSeTFWCgJyGwGJEICthyB_KbkMoguELFyVnAQZE/edit?tab=t.0",
    "https://docs.google.com/document/d/1nYnhwzU9mpMcojRXz90somZv9oUxhB_D-C-adh4bpQc/edit?tab=t.0",
    "https://docs.google.com/document/d/177XPkHZBX4hqn5D0lleQhM9UmG88h0WUdaLl2jRkbC0/edit?tab=t.0",
    "https://docs.google.com/document/d/186KkxhDSi3VpDnyAu06ktKGAoLJxzCeJBmBAmX0noNQ/edit?tab=t.0",
    "https://deepwoken.fandom.com/wiki/Bosses",
]

# 키 로드 및 자동 순환
key_list = [k.strip() for k in API_KEYS.split(',') if k.strip()]
if not key_list:
    print("⚠️ API_KEYS 변수에 Gemini API 키를 입력해 주세요!")
    sys.exit(0)

current_key_idx = 0
genai.configure(api_key=key_list[0])
print(f"🔑 {len(key_list)}개의 API 키가 로드되었습니다.")

def rotate_key():
    global current_key_idx
    if len(key_list) > 1:
        current_key_idx = (current_key_idx + 1) % len(key_list)
        print(f"🔄 [Key Rotation] 다음 API 키로 전환 (Key #{current_key_idx+1}/{len(key_list)})")
        genai.configure(api_key=key_list[current_key_idx])

print(f"🎯 총 {len(URL_LIST)}개의 링크를 순차적으로 분석합니다...\n" + "="*60)

for i, url in enumerate(URL_LIST, 1):
    print(f"\n⚡ [{i}/{len(URL_LIST)}] 분석 중: {url}")
    try:
        if "youtube.com" in url or "youtu.be" in url:
            vid_match = re.search(r'(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})', url)
            vid_id = vid_match.group(1) if vid_match else f"yt_{int(time.time())}"
            target_file = f"data/videos/{vid_id}.mp4"
            
            ydl_opts = {
                'format': 'bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]/best',
                'outtmpl': f'data/videos/{vid_id}.%(ext)s',
                'merge_output_format': 'mp4',
                'quiet': True
            }
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=True)
                title = info.get('title', 'YouTube Build')
            
            print(f"  -> 영상 업로드 및 Gemini AI 분석 중...")
            vfile = genai.upload_file(target_file)
            while vfile.state.name == "PROCESSING":
                time.sleep(3)
                vfile = genai.get_file(vfile.name)
            
            model = genai.GenerativeModel("gemini-2.5-flash-lite", generation_config={"response_mime_type": "application/json"})
            prompt = "Analyze this Deepwoken build video and return a JSON object with: build_summary, stats_and_attunements, talents_and_mantras, equipment, shrine_of_order_progression, combo_and_playstyle."
            res = model.generate_content([vfile, prompt])
            
            with open(f"data/analysis/{vid_id}.json", "w", encoding="utf-8") as f:
                f.write(res.text)
            with open(f"data/knowledge_base/{vid_id}.md", "w", encoding="utf-8") as f:
                f.write(f"# ⚔️ {title}\n\n- URL: {url}\n\n```json\n{res.text}\n```")
            print(f"  ✅ [완료] {title}")
            try: os.remove(target_file)
            except: pass
            
        elif "docs.google.com" in url:
            doc_id = re.search(r'docs\.google\.com/document/d/([a-zA-Z0-9_-]+)', url).group(1)
            req = urllib.request.Request(f"https://docs.google.com/document/d/{doc_id}/export?format=txt", headers={'User-Agent': 'Mozilla/5.0'})
            text = urllib.request.urlopen(req, timeout=10).read().decode('utf-8', errors='replace')[:30000]
            
            print(f"  -> 텍스트({len(text)}자) Gemini AI 분석 중...")
            model = genai.GenerativeModel("gemini-2.5-flash-lite", generation_config={"response_mime_type": "application/json"})
            res = model.generate_content(f"Extract Deepwoken build details, stats, mantras, talents into structured JSON:\n\n{text}")
            
            with open(f"data/analysis/{doc_id}.json", "w", encoding="utf-8") as f:
                f.write(res.text)
            with open(f"data/knowledge_base/{doc_id}.md", "w", encoding="utf-8") as f:
                f.write(f"# 📜 Google Docs Guide ({doc_id})\n\n- URL: {url}\n\n```json\n{res.text}\n```")
            print(f"  ✅ [완료] Google Docs Guide")
            
        else:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            html = urllib.request.urlopen(req, timeout=10).read().decode('utf-8', errors='replace')
            soup = BeautifulSoup(html, 'html.parser')
            text = soup.get_text()[:25000]
            
            model = genai.GenerativeModel("gemini-2.5-flash-lite", generation_config={"response_mime_type": "application/json"})
            res = model.generate_content(f"Summarize Deepwoken build / wiki into structured JSON:\n\n{text}")
            slug = re.sub(r'[^a-zA-Z0-9_-]', '_', url.split('/')[-1])
            with open(f"data/analysis/{slug}.json", "w", encoding="utf-8") as f:
                f.write(res.text)
            print(f"  ✅ [완료] Web ({url})")
            
    except Exception as e:
        print(f"  ⚠️ 오류 발생: {e}")
        if "429" in str(e) or "quota" in str(e).lower():
            rotate_key()
    
    time.sleep(3)

print("\n" + "="*60 + "\n🎉 10개 모든 링크의 분석이 완료되었습니다!")
# 결과 파일 압축 다운로드
!zip -r -q deepwoken_builds_result.zip data/analysis data/knowledge_base
from google.colab import files
files.download('deepwoken_builds_result.zip')